# GDPO.py 51~100줄 주석

**날짜**: 2026-04-16  
**주제**: GDPO Base 클래스 - Reward, Tool Reward, Reasoning Judge, Memory 설정

## 오늘 배운 것

- `self.gdpo_config.get(key, default)` 패턴으로 설정 안전 접근
- 조건부 보상(conditioned rewards) vs 기본 보상 차이
- Tool Reward: 도구 사용 보상 시스템
- Reasoning Judge: 추론 품질 심사 (기본 비활성화)
- Sequential 모드: 메모리 절약을 위한 순차 처리

In [ ]:
import torch
from typing import Tuple, Optional, List

# ================================================
# GDPO.py 51~100줄 주석
# ================================================

# 실행 가능하게 하기 위한 placeholder 클래스
class GDPOAgent:
    def __init__(self, gdpo_config, tokenizer, max_new_tokens, G):
        self.gdpo_config = gdpo_config
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens
        self.G = G

        # Reward config / 보상(Reward) 관련 설정
        self.use_conditioned_rewards = self.gdpo_config.get("use_conditioned_rewards", False)
                              # 조건부 보상 여부. 기본 False
        self.accuracy_threshold = self.gdpo_config.get("accuracy_threshold", 1.0)
                              # 정확도 기준점. 기본 1.0(100%)
        self.target_length = self.gdpo_config.get("target_length", 1024)
                              # 목표 출력 길이(토큰수). 기본 1024

        # Tool Reward config / 도구 사용 보상 설정 (GDPO 논문. 기본값: 활성화)
        self.enable_tool_reward = self.gdpo_config.get("enable_tool_reward", True)
                              # 도구 사용 보상 활성화 여부. 기본 True(켜짐)
        self.tool_correctness_threshold = self.gdpo_config.get("tool_correctness_threshold", 1.5)
                              # 도구 정답 판단 기준점. 기본 1.5

        # Reasoning Judge config / 추론 품질 심사 설정
        self.enable_reasoning_judge = self.gdpo_config.get("enable_reasoning_judge", False)
                              # 추론 품질 심사 사용 여부. 기본 False
        self.reasoning_judge = None  # 심사 도구. 아직 없으므로 None으로 초기화
        self.reasoning_quality_threshold = 0.5
                              # 추론 품질 합격 기준. 0.5 (50점 이상이면 통과)

        if self.enable_reasoning_judge:  # 만약 추론 심사가 켜져 있다면
            # from utils.reasoning_judge import ReasoningJudge
            # self.reasoning_judge = ReasoningJudge()
            print("ReasoningJudge is enabled")
                              # 심사 도구를 실제로 만들어서 저장

        # Memory Optimization config / 메모리 최적화 설정
        self.sequential = self.gdpo_config.get("sequential", False)
                              # 순차 처리 모드. True면 메모리 절약, 기본 False

    def generate_samples(            # "샘플을 생성하라"는 기능(함수)을 만든다
        self,                         # 이 상자(클래스) 자신
        model: torch.nn.Module,       # AI 언어 모델(생성에 사용)
        input_ids: torch.Tensor,      # 입력 텍스트를 숫자로 바꾼 것 (B, seq_len) 형태
        attention_mask: torch.Tensor  # 어떤 토큰을 봐야 하는지 표시 (B, seq_len) 형태
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], int]:
                                      # 반환값: (생성된 시퀀스, 온도 보상값 또는 None, 실제 그룹 크기)
        """
        온도 대비 샘플링 옵션이 있는 샘플 생성 함수

        Args(입력값):
          model: 언어 모델
          input_ids: 입력 토큰 ID (B=배치크기, seq_len=문장길이)
          attention_mask: 어텐션 마스크

        Returns(반환값):
          sequences: 생성된 시퀀스 (B * effective_G, seq_len)
          temperature_rewards: 온도 라벨 또는 None
          effective_G: 실제 그룹 크기 (G 또는 2G)
        """
        gen_kwargs = {                            # 생성(generation)에 필요한 설정값들을 딕셔너리로 묶음
            "max_new_tokens": self.max_new_tokens,  # 최대 생성 토큰 수
            "do_sample": True,                    # True = 확률적으로 다양하게 생성(False면 항상 같은 결과)
            "top_p": 0.95,                        # 상위 95% 확률 토큰 중에서만 선택 (nucleus sampling)
            "pad_token_id": self.tokenizer.pad_token_id,
                                                  # 문장 길이 맞추기용 패딩 토큰 ID
            "eos_token_id": self.tokenizer.eos_token_id,
                                                  # 문장 끝을 알리는 토큰 ID
            "num_return_sequences": self.G,       # 한 입력당 G개의 답변을 생성
            "use_cache": False,                   # 캐시 사용 안 함 (메모리 절약)
        }
        # 생성 로직 placeholder
        return torch.empty(1), None, 1

## 회고

- `getattr`와 `dict.get(key, default)` 패턴으로 기본값 지정
- 도구 사용 보상 개념이 흥미로움 (AI가 도구를 잘 썼는지 평가)
- 추론 품질 심사는 아직 복잡해서 기본 비활성화